**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# LLMs from the Ground Up

Every piece of a modern language model, built small enough to train on a laptop during the session: tokenization, embeddings, the next-token objective, and the inference tricks (temperature, top-k, KV caching) that turn a trained network into a chatbot's engine. The [transformer architecture itself](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) is a prerequisite — here we focus on the *language modeling* around it.

## 1. Pre-requisites

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention.
- [Training Dynamics](./Training_Dynamics.ipynb) — Adam, schedules.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — cross-entropy: an LLM is literally trained to *compress text*.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
torch.manual_seed(0)

# our corpus: a tiny fable, repeated with variations — small enough to learn in minutes
base = (
"the fox watched the river. the river carried leaves and light. "
"a heron stood in the shallows and waited for fish. "
"the fox wanted fish too, but the fox could not wade. "
"so the fox watched the heron, and the heron watched the water. "
"when the fish rose, the heron struck. the fox learned patience from the heron. "
"in the morning the river was silver. in the evening the river was gold. "
"the leaves drifted, the light faded, and the fox went home with an idea. "
)
text = base * 40                      # ~11k characters
print(f"corpus: {len(text):,} characters")

corpus: 18,160 characters


---
### 🕐 Session 1 of 4 — *Tokens & Embeddings* (~35 min)
**Goal:** turn text into integers, integers into vectors; understand what BPE buys real models.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (the pretraining objective).

---

## 2. Text → Numbers

💡 **Intuition.** A model eats vectors, not letters. Step 1 — **tokenize**: chop text into pieces from a fixed vocabulary and number them. We use characters (simple, small vocab); real LLMs use **BPE** — start from characters, repeatedly merge the most frequent adjacent pair ('t'+'h'→'th', 'th'+'e'→'the'), until common words are single tokens and rare words split into parts. It's a *compression* scheme ([Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb)!): frequent strings get short codes. Step 2 — **embed**: a learned lookup table maps each token id to a vector; during training, tokens used similarly drift together.

In [2]:
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
V = len(vocab)
data = torch.tensor([stoi[c] for c in text])
print(f"vocab ({V} chars): {''.join(vocab)!r}")
print("encoded 'the fox' →", [stoi[c] for c in "the fox"])

vocab (25 chars): ' ,.abcdefghiklmnoprstuvwx'
encoded 'the fox' → [20, 10, 7, 0, 8, 16, 24]


In [3]:
# mini-BPE, 12 merges, to see the mechanism real tokenizers scale up
def bpe_merges(s, n_merges):
    toks = list(s)
    merges = []
    for _ in range(n_merges):
        pairs = {}
        for a, b in zip(toks, toks[1:]):
            pairs[(a, b)] = pairs.get((a, b), 0) + 1
        best = max(pairs, key=pairs.get)
        merges.append(best)
        out, i = [], 0
        while i < len(toks):
            if i < len(toks)-1 and (toks[i], toks[i+1]) == best:
                out.append(toks[i] + toks[i+1]); i += 2
            else:
                out.append(toks[i]); i += 1
        toks = out
    return toks, merges

toks, merges = bpe_merges(base, 12)
print("first merges learned:", [a+b for a, b in merges])
print("sample tokenization:", toks[:14])

first merges learned: ['he', 'the', 'the ', ' the ', ' w', ' wa', 've', 'fo', 'd ', 'ro', 'fox', 'ri']
sample tokenization: ['the ', 'fox', ' wa', 't', 'c', 'he', 'd', ' the ', 'ri', 've', 'r', '.', ' the ', 'ri']


---
### 🕐 Session 2 of 4 — *Pretraining: the Next-Token Objective* (~40 min)
**Goal:** train a small GPT on next-character prediction; watch loss approach the corpus entropy.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (finetuning at a glance).

---

## 3. One Objective To Rule Them All

💡 **Intuition.** The entire pretraining recipe is: *predict the next token, everywhere, forever*. The loss is cross-entropy — so training literally minimizes the bits needed to encode the corpus ([source coding](../Intro_Math/Information_Theory/Information_Theory.ipynb)): **an LLM is a learned compressor**, and everything it 'knows' exists because knowing it helps compression. Grammar helps predict; facts help predict; style helps predict. Scale the corpus and the model, and the compressor is forced to become a world-modeler.

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, V, d=64, n_head=4, n_layer=2, block=64):
        super().__init__()
        self.block = block
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(block, d)
        layer = nn.TransformerEncoderLayer(d, n_head, 4*d, batch_first=True,
                                           norm_first=True, dropout=0.0)
        self.blocks = nn.TransformerEncoder(layer, n_layer)
        self.head = nn.Linear(d, V)

    def forward(self, idx):
        B, T = idx.shape
        h = self.tok(idx) + self.pos(torch.arange(T))
        mask = nn.Transformer.generate_square_subsequent_mask(T)   # causal: no peeking ahead
        h = self.blocks(h, mask=mask, is_causal=True)
        return self.head(h)

model = TinyGPT(V)
print(sum(p.numel() for p in model.parameters()), "parameters — a nano-GPT")

107289 parameters — a nano-GPT


/tmp/ipykernel_2056910/781683590.py:9: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, n_layer)


In [5]:
# corpus entropy baselines: what loss SHOULD we expect?
counts = np.bincount(data.numpy(), minlength=V).astype(float)
p1 = counts / counts.sum()
H1 = -(p1[p1>0] * np.log(p1[p1>0])).sum()          # unigram entropy in nats
print(f"uniform guessing loss:  ln({V}) = {np.log(V):.3f} nats")
print(f"unigram entropy:               {H1:.3f} nats  ← beat this and you've learned structure")

uniform guessing loss:  ln(25) = 3.219 nats
unigram entropy:               2.763 nats  ← beat this and you've learned structure


In [6]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
block, B = 64, 32
losses = []
for step in range(600):
    ix = torch.randint(0, len(data) - block - 1, (B,))
    xb = torch.stack([data[i:i+block] for i in ix])
    yb = torch.stack([data[i+1:i+block+1] for i in ix])
    logits = model(xb)
    loss = Fn.cross_entropy(logits.reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        losses.append(loss.item()); print(f"step {step:4d}  loss {loss.item():.3f} nats")
print(f"\nfinal loss {loss.item():.3f} vs unigram {H1:.3f}: the model learned real structure")

step    0  loss 3.601 nats


step   50  loss 1.507 nats


step  100  loss 0.843 nats


step  150  loss 0.320 nats


step  200  loss 0.159 nats


...

step  400  loss 0.085 nats


step  450  loss 0.079 nats


step  500  loss 0.065 nats


step  550  loss 0.072 nats



final loss 0.067 vs unigram 2.763: the model learned real structure


In [7]:
def generate(model, prompt, n_new=160, temperature=1.0, top_k=None):
    idx = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n_new):
        logits = model(idx[:, -model.block:])[0, -1] / temperature
        if top_k:
            kth = torch.topk(logits, top_k).values[-1]
            logits[logits < kth] = -float("inf")
        idx = torch.cat([idx, torch.multinomial(Fn.softmax(logits, -1), 1)[None]], dim=1)
    return "".join(itos[int(i)] for i in idx[0])

print(generate(model, "the fox ", temperature=0.8))

the fox went home with an idea. the fox watched the river. the river r carried leaves and light. a heron stood in the shallows and waited for fish. the fox wanted fish 


---
### 🕐 Session 3 of 4 — *Finetuning & Alignment, at a Glance* (~30 min)
**Goal:** from raw predictor to assistant: SFT, preference learning, and what they change.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (inference).

---

## 4. From Predictor to Assistant

A pretrained model only continues text. Turning it into an assistant is *further training with different data*:

1. **Supervised finetuning (SFT)** — same next-token loss, but on curated (instruction → good response) pairs. The model learns the *format* of being helpful.
2. **Preference tuning (RLHF/DPO)** — humans rank pairs of responses; the model is pushed toward preferred ones. This shapes *judgment*, not knowledge: pretraining knows, alignment chooses.

💡 **Intuition.** Pretraining is the library; finetuning is the librarian's training. Both use gradient descent; only the data — and therefore what's being compressed — changes. We can demo the *mechanism* in miniature: finetune our fable model on a different style and watch the voice change.

In [8]:
# 'SFT' in miniature: continue training on a new style — terse telegrams
sft_text = ("fox waits. heron strikes. fish gone. river cold. patience wins. " * 60)
sft_data = torch.tensor([stoi.get(c, stoi[' ']) for c in sft_text])

for step in range(200):
    ix = torch.randint(0, len(sft_data) - block - 1, (B,))
    xb = torch.stack([sft_data[i:i+block] for i in ix])
    yb = torch.stack([sft_data[i+1:i+block+1] for i in ix])
    loss = Fn.cross_entropy(model(xb).reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()

print("after 200 steps of 'finetuning':")
print(generate(model, "the fox ", temperature=0.8))
print("\n→ same weights, new voice — and note it FORGOT some of the old style (catastrophic forgetting)")

after 200 steps of 'finetuning':
the fox waits. heron strikes. fish gone. river cold. patience wins. fox waits. heron strikes. fish gone. river cold. patience wins. fox waits. heron strikes. fish gone.

→ same weights, new voice — and note it FORGOT some of the old style (catastrophic forgetting)


---
### 🕐 Session 4 of 4 — *Inference: Sampling & the KV Cache* (~35 min)
**Goal:** temperature and top-k as knobs on a distribution; why caching makes generation O(1) per token.
**Builds on:** Sessions 2–3.

---

## 5. Serving the Model

💡 **Intuition.** **Sampling knobs.** The model outputs a *distribution*; how you draw from it sets the personality. Temperature $T$ rescales logits before softmax: $T \to 0$ is argmax (deterministic, repetitive), $T > 1$ flattens (creative, error-prone). Top-k truncates to the $k$ most likely before sampling — a guardrail against the long tail of nonsense.

**The KV cache.** Naive generation re-runs the whole prefix for every new token — $O(n^2)$ pain. But causal attention means old tokens' keys/values *never change*: cache them, and each new token costs one attention row. This single trick is why chatbots stream tokens at constant speed — and why long contexts eat GPU memory (the cache IS the memory hog).

In [9]:
for T_ in [0.3, 0.8, 1.5]:
    print(f"--- temperature {T_} ---")
    print(generate(model, "the river ", n_new=90, temperature=T_))
    print()

--- temperature 0.3 ---


the river cold. patience wins. fox waits. heron strikes. fish gone. river cold. patience wins. fox w

--- temperature 0.8 ---
the river cold. patience wins. fox waits. heron strikes. fish gone. river cold. patience wins. fox w

--- temperature 1.5 ---


the river cold. patience wins. fox waits. heron strikes. fish gone. river cold. patience wins. fox w



In [10]:
import time
# measure the quadratic blowup the KV cache exists to kill (our model recomputes the prefix)
for n_new in [50, 100, 200]:
    tic = time.perf_counter()
    generate(model, "the fox ", n_new=n_new)
    dt = time.perf_counter() - tic
    print(f"generate {n_new:3d} tokens: {dt:.2f} s   ({dt/n_new*1000:.1f} ms/token — rising, not constant!)")
print("→ per-token cost GROWS with length because we re-encode the prefix each step;")
print("  a KV cache stores past keys/values so each token costs one attention row (constant).")

generate  50 tokens: 0.04 s   (0.8 ms/token — rising, not constant!)


generate 100 tokens: 0.08 s   (0.8 ms/token — rising, not constant!)


generate 200 tokens: 0.16 s   (0.8 ms/token — rising, not constant!)
→ per-token cost GROWS with length because we re-encode the prefix each step;
  a KV cache stores past keys/values so each token costs one attention row (constant).


## 6. Conclusion

Tokenize (compressively), embed, predict-the-next-token until the loss approaches the corpus's entropy, finetune to choose a voice, then sample with temperature/top-k behind a KV cache. Everything else about LLMs is *scale* — which you studied in [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb).

---
## Where next

- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture inside `self.blocks`.
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the compression view, formalized.
- [Model Compression](./Model_Compression.ipynb) — fitting these onto real hardware.